# Harry Potter 2 — Nettoyage des données & Préparation Gephi
**Corpus 1 :** `Harry-Potter-and-the-Chamber-of-Secrets.pdf` (roman)  
**Corpus 2 :** `harry-potter-and-the-chamber-of-secrets-2002.pdf` (script — PDF texte natif)

Ce notebook nettoie les deux corpus et produit les fichiers prêts pour Gephi :
- `novel_nodes.csv` / `novel_edges.csv`
- `screenplay_nodes.csv` / `screenplay_edges.csv`

> **Différence importante avec l'ébauche de projet** : je voulais travailler sur le tome 1 mais le script étaint en scan donc très compliqué à extraire en texte utilisable numériquement.

---

## 0. Imports

In [1]:
import re
import os
import csv
import collections
import pandas as pd
from pdfminer.high_level import extract_text

os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs/chapters', exist_ok=True)
print('Tout est importé correctement')

Tout est importé correctement


---
## 1. Roman — Nettoyage du PDF

Le PDF du roman possède une couche texte embarquée (même éditeur que le tome 1),  
aucun OCR n'est donc nécessaire. `pdfminer` l'extrait directement.

### 1.1 Extraction brute 

In [2]:
# pdfminer lit la couche texte embarquée dans le PDF — pas d'OCR nécessaire

raw_novel = extract_text('Harry-Potter-and-the-Chamber-of-Secrets.pdf')
print(f'Texte extrait : {len(raw_novel.split()):,} mots bruts')

Texte extrait : 87,933 mots bruts


### 1.2 Suppression du front matter

In [3]:
# Le PDF commence par plusieurs infos non essentielles dont la page de copyright, la dédicace, etc.
# Le chapitre 1 commence avec le motif "— CHAPTER ONE —"

match = re.search(r'—\s*CHAPTER ONE\s*—', raw_novel)
if match:
    novel_text = raw_novel[match.start():]
    print('Suppression réussie => le texte commence à CHAPTER ONE')
else:
    novel_text = raw_novel
    print('Suppression échouée : marqueur de chapitre introuvable => utilisation du texte complet')

Suppression réussie => le texte commence à CHAPTER ONE


### 1.3 Normalisation

In [4]:
# pdfminer produit parfois des caractères de saut de page (\x0c) aux sauts de page
# ainsi que des espaces multiples résiduels

novel_text = novel_text.replace('\x0c', '\n')        # saut de page -> retour à la ligne
novel_text = re.sub(r' {2,}', ' ', novel_text)         # espaces multiples -> un seul
novel_text = re.sub(r'\n{3,}', '\n\n', novel_text)  # lignes vides excessives

print(f'Texte nettoyé : {len(novel_text.split()):,} mots')
print('\n--- Aperçu (400 premiers caractères) ---')
print(novel_text[:400])

Texte nettoyé : 87,616 mots

--- Aperçu (400 premiers caractères) ---
— CHAPTER ONE — 

The Worst Birthday 

Not for the first time, an argument had broken out over breakfast 
at number four, Privet Drive. Mr Vernon Dursley had been woken 
in the early hours of the morning by a loud, hooting noise from his 
nephew Harry’s room. 

‘Third time this week!’ he roared across the table. ‘If you can’t 

control that owl, it’ll have to go!’ 

Harry tried, yet again, to expl


### 1.4 Découpage par chapitres

In [5]:
# Les en-têtes de chapitres suivent le motif :  — CHAPTER [MOT]+ — suivi d'une ligne vide puis du titre du chapitre
# re.split avec un groupe capturant alterne [avant, titre1, texte1, titre2, texte2 ...]

CHAP_PATTERN = r'(—\s*CHAPTER [A-Z]+\s*—\s*\n\n[^\n]+)'
titles  = re.findall(CHAP_PATTERN, novel_text)
parts   = re.split(CHAP_PATTERN, novel_text)

chapters = {}
for i, title in enumerate(titles):
    chapter_text = parts[2 * i + 2].strip()
    chapters[title.strip()] = chapter_text

print(f"{len(chapters)} chapitres détectés :")

for title, text in chapters.items():
    title_propre = re.sub(r"\n\s*\n", "\n", title.strip())
    print(f"{title_propre} ({len(text.split()):,} mots)")
    print()

18 chapitres détectés :
— CHAPTER ONE — 
The Worst Birthday (2,645 mots)

— CHAPTER TWO — 
Dobby’s Warning (2,985 mots)

— CHAPTER THREE — 
The Burrow (4,606 mots)

— CHAPTER FOUR — 
At Flourish and Blotts (5,938 mots)

— CHAPTER FIVE — 
The Whomping Willow (5,458 mots)

— CHAPTER SIX — 
Gilderoy Lockhart (4,679 mots)

— CHAPTER SEVEN — 
Mudbloods and Murmurs (4,654 mots)

— CHAPTER EIGHT — 
The Deathday Party (4,527 mots)

— CHAPTER NINE — 
The Writing on the Wall (5,309 mots)

— CHAPTER TEN — 
The Rogue Bludger (5,438 mots)

— CHAPTER ELEVEN — 
The Duelling Club (6,134 mots)

— CHAPTER TWELVE — 
The Polyjuice Potion (5,545 mots)

— CHAPTER THIRTEEN — 
The Very Secret Diary (5,622 mots)

— CHAPTER FOURTEEN — 
Cornelius Fudge (4,038 mots)

— CHAPTER FIFTEEN — 
Aragog (4,702 mots)

— CHAPTER SIXTEEN — 
The Chamber of Secrets (5,838 mots)

— CHAPTER SEVENTEEN — 
The Heir of Slytherin (5,570 mots)

— CHAPTER EIGHTEEN — 
Dobby’s Reward (3,803 mots)



### 1.5 Export des chapitres

In [6]:
# On exporte les chapitres AVANT la suppression de ponctuation car le titre contient des tirets et caractères nécessaires à la lisibilité

for i, (title, text) in enumerate(chapters.items(), 1):
    with open(f'outputs/chapters/chap_{i:02d}.txt', 'w', encoding='utf-8') as f:
        f.write(f'{title}\n\n{text}')

df_novel = pd.DataFrame([
    {'chapter_num': i, 'title': title, 'text': text}
    for i, (title, text) in enumerate(chapters.items(), 1)
])
df_novel.to_csv('outputs/novel_chapters.csv', index=False, encoding='utf-8')

print('  outputs/chapters/chap_XX.txt  (un par chapitre)')
print('  outputs/novel_chapters.csv    (-> Orange / OpenRefine)')

  outputs/chapters/chap_XX.txt  (un par chapitre)
  outputs/novel_chapters.csv    (-> Orange / OpenRefine)


### 1.6 Suppression ponctuation et export

In [7]:
novel_text = re.sub(r"[^\w\s\-]", ' ', novel_text)
novel_text = re.sub(r' {2,}', ' ', novel_text)

with open('outputs/novel_cleaned.txt', 'w', encoding='utf-8') as f:
    f.write(novel_text)

print(f'Texte roman final : {len(novel_text.split()):,} mots')
print('\n--- Aperçu ---')
print(novel_text[:400])

Texte roman final : 89,192 mots

--- Aperçu ---
 CHAPTER ONE 

The Worst Birthday 

Not for the first time an argument had broken out over breakfast 
at number four Privet Drive Mr Vernon Dursley had been woken 
in the early hours of the morning by a loud hooting noise from his 
nephew Harry s room 

 Third time this week he roared across the table If you can t 

control that owl it ll have to go 

Harry tried yet again to explain 
 She s bored


---
## 2. Script — Nettoyage du PDF texte natif

Le script du tome 2 est un **PDF généré depuis Word** (contrairement au tome 1 qui était un scan OCR).  
Le bruit à supprimer est structurel : en-têtes de page, numéros de scène, numéros de page, lignes `CONTINUED:`, indications de mise en scène (`FADE IN`, `DISSOLVE TO`, etc.).

### 2.1 Extraction brute

In [8]:
# pdfminer lit la couche texte Word embarquée — texte propre, pas d'OCR

raw_screenplay = extract_text('harry-potter-and-the-chamber-of-secrets-2002.pdf')
print(f'Texte extrait : {len(raw_screenplay.split()):,} mots bruts')
print('\n--- Aperçu (600 premiers caractères) ---')
print(raw_screenplay[:600])

Texte extrait : 28,248 mots bruts

--- Aperçu (600 premiers caractères) ---
Rev. 03/01/02 (Buff)
Rev. 03/14/02 (Salmon)
Rev. 04/02/02 (Cherry)
Rev. 04/02/02 (Tan)
    (not distributed)
Rev. 04/03/02 (2nd White)
Rev. 04/03/02 (2nd Blue)
Rev. 04/09/02 (2nd Pink)
Rev. 04/16/02 (2nd Yellow)
Rev. 04/17/02 (2nd Green)
Rev. 05/14/02 (2nd Gold)

HARRY POTTER AND
THE CHAMBER OF SECRETS

screenplay by STEVEN KLOVES

based on the novel by
J.K. ROWLING

No portion of this script may be performed, reproduced,
or used by any means, or quoted or published in any
medium without the prior written consent of Warner Bros.

WARNER BROS.
4000 Warner Boulevard
Burbank,  California  91522




### 2.2 Suppression du front matter

In [9]:
# Le PDF commence par la page de titre (liste des révisions, mention légale, etc.)
# On localise le premier "FADE IN:" qui marque le début du script, en gros on supprime la première page.

match_debut = re.search(r'\bFADE IN:', raw_screenplay)
screenplay_text = raw_screenplay[match_debut.start():] if match_debut else raw_screenplay

print(f'Après suppression du front matter : {len(screenplay_text.split()):,} mots')
print('\n--- Aperçu du début ---')
print(screenplay_text[:400])

Après suppression du front matter : 28,148 mots

--- Aperçu du début ---
FADE IN:

1

EXT. PRIVET DRIVE - DAY

WIDE HELICOPTER SHOT.  Privet Drive.  CAMERA CRANES DOWN,
DOWN, OVER the rooftops, FINDS the SECOND FLOOR WINDOW of
NUMBER 4.  HARRY POTTER sits in the window.

2

3

OMITTED

INT. HARRY'S BEDROOM - DAY

Harry pages through a SCRAPBOOK, stops on a MOVING PHOTO
of Ron and Hermione.  SQUAWK!  Harry jumps.  HEDWIG pecks
at the LOCK slung through her cage door, th


### 2.3 Suppression du bruit structurel 

In [10]:
# On supprime les éléments de mise en page du format screenplay.

# En-têtes de page répétitifs (91 occurrences) : "THE CHAMBER OF SECRETS - Rev. X/XX/XX"
screenplay_text = re.sub(
    r'^THE CHAMBER OF SECRETS\s*[-–]\s*Rev\.[^\n]*$',
    '', screenplay_text, flags=re.MULTILINE
)

# Numéros de page autonomes (ex : "2.", "13.", "135.")
screenplay_text = re.sub(r'^\s*\d+\.\s*$', '', screenplay_text, flags=re.MULTILINE)

# Numéros de scène seuls sur une ligne (ex : "1", "13A", "135A", "14\nthru\n16")
screenplay_text = re.sub(r'^\s*\d+[A-Z]?\s*$', '', screenplay_text, flags=re.MULTILINE)
screenplay_text = re.sub(r'^\s*thru\s*$', '', screenplay_text, flags=re.MULTILINE)

# Lignes CONTINUED: et CONTINUED: (2)
screenplay_text = re.sub(r'^\s*CONTINUED:\s*(\(\d+\))?\s*$', '', screenplay_text, flags=re.MULTILINE)

# Scènes supprimées : "OMITTED"
screenplay_text = re.sub(r'^.*OMITTED.*$', '', screenplay_text, flags=re.MULTILINE)

# Parenthétiques seuls sur une ligne : (CONT'D), (O.S.), (V.O.), (MORE), etc.
screenplay_text = re.sub(r'^\s*\([^)]{1,60}\)\s*$', '', screenplay_text, flags=re.MULTILINE)

# Transitions de mise en scène
screenplay_text = re.sub(
    r'^(FADE IN:|FADE TO BLACK\.|DISSOLVE TO:|CUT TO:|SMASH CUT TO:|CAMERA[^\n]*)$',
    '', screenplay_text, flags=re.MULTILINE
)

# Indications de scène EXT./INT. (et combinaisons comme EXT./INT.)
screenplay_text = re.sub(
    r'^\s*\d*[A-Z]?\s*(EXT\.|INT\.|EXT\./INT\.|INT\./EXT\.)[^\n]*$',
    '', screenplay_text, flags=re.MULTILINE
)

# Lignes vides excessives
screenplay_text = re.sub(r'\n{3,}', '\n\n', screenplay_text)

print(f'Après suppression du bruit : {len(screenplay_text.split()):,} mots')
print('\nAperçu : ')
print(screenplay_text[:600])

Après suppression du bruit : 26,005 mots

Aperçu : 


WIDE HELICOPTER SHOT.  Privet Drive.  CAMERA CRANES DOWN,
DOWN, OVER the rooftops, FINDS the SECOND FLOOR WINDOW of
NUMBER 4.  HARRY POTTER sits in the window.

Harry pages through a SCRAPBOOK, stops on a MOVING PHOTO
of Ron and Hermione.  SQUAWK!  Harry jumps.  HEDWIG pecks
at the LOCK slung through her cage door, then glowers at
Harry.

HARRY
I can't, Hedwig.  I'm not allowed
to use magic outside of school.
Besides, if Uncle Vernon --

At the sound of the name, HEDWIG SQUAWKS again, LOUDER.

Har-ry Pot-ter!

UNCLE VERNON (O.S.)

HARRY
Now you've done it.

While AUNT PETUNIA puts the finish


### 2.4 Vérification 

In [11]:
# Le script du tome 2 est généré depuis Word d'après mes recherches : les noms sont correctement orthographiés. 
#On fait donc simplement un contrôle de cohérence en comptant les personnages principaux.

print('Vérification des occurrences des personnages principaux :')
for name in ['Harry', 'Ron', 'Hermione', 'Hagrid', 'Dumbledore', 'Dobby', 'Malfoy', 'Ginny']:
    count = len(re.findall(r'\b' + name + r'\b', screenplay_text, re.IGNORECASE))
    print(f'  {name} : {count} occurrences')

Vérification des occurrences des personnages principaux :
  Harry : 815 occurrences
  Ron : 423 occurrences
  Hermione : 181 occurrences
  Hagrid : 112 occurrences
  Dumbledore : 95 occurrences
  Dobby : 106 occurrences
  Malfoy : 96 occurrences
  Ginny : 58 occurrences


### 2.5 Suppression ponctuation et export 

In [12]:
screenplay_text = re.sub(r"[^\w\s\-]", ' ', screenplay_text)
screenplay_text = re.sub(r' {2,}', ' ', screenplay_text)

with open('outputs/screenplay_cleaned.txt', 'w', encoding='utf-8') as f:
    f.write(screenplay_text)

print(f'Texte script final : {len(screenplay_text.split()):,} mots')
print('\n--- Aperçu ---')
print(screenplay_text[:400])

Texte script final : 27,196 mots

--- Aperçu ---


WIDE HELICOPTER SHOT Privet Drive CAMERA CRANES DOWN 
DOWN OVER the rooftops FINDS the SECOND FLOOR WINDOW of
NUMBER 4 HARRY POTTER sits in the window 

Harry pages through a SCRAPBOOK stops on a MOVING PHOTO
of Ron and Hermione SQUAWK Harry jumps HEDWIG pecks
at the LOCK slung through her cage door then glowers at
Harry 

HARRY
I can t Hedwig I m not allowed
to use magic outside of school 
Besi


---
## 3. Préparation pour Gephi

On construit les réseaux de cooccurrence des personnages pour les deux corpus.  
Deux personnages **cooccurrent** s'ils apparaissent dans une fenêtre de **100 mots** l'un de l'autre.

Pour chaque corpus on produit :
- **`_nodes.csv`** — liste des personnages avec leur nombre de mentions et leur maison
- **`_edges.csv`** — paires cooccurrentes avec un poids

### 3.1 Dictionnaire des personnes et leurs alias 

In [13]:
CHARACTERS = {
    'Harry Potter':             ['Harry', 'Potter'],
    'Hermione Granger':         ['Hermione', 'Granger'],
    'Ronald Weasley':           ['Ron', 'Ronald', 'Ron Weasley'],
    'Ginny Weasley':            ['Ginny'],
    'Albus Dumbledore':         ['Dumbledore', 'Albus'],
    'Lord Voldemort':           ['Voldemort', 'You-Know-Who', 'He-Who-Must-Not-Be-Named', 'Dark Lord', 'Tom Riddle', 'Riddle'],
    'Neville Longbottom':       ['Neville', 'Longbottom'],
    'Draco Malfoy':             ['Draco', 'Malfoy'],
    'Severus Snape':            ['Snape', 'Severus'],
    'Rubeus Hagrid':            ['Hagrid', 'Rubeus'],
    'Vernon Dursley':           ['Vernon', 'Uncle Vernon'],
    'Petunia Dursley':          ['Petunia', 'Aunt Petunia'],
    'Dudley Dursley':           ['Dudley'],
    'Dobby':                    ['Dobby'],
    'Cornelius Fudge':          ['Fudge', 'Cornelius'],
    'Lucius Malfoy':            ['Lucius'],
    'Narcissa Malfoy':          ['Narcissa'],
    'Mr Borgin':                ['Borgin'],
    'Mrs Norris':               ['Norris', 'Mrs Norris'],
    'Argus Filch':              ['Filch', 'Argus'],
    'Moaning Myrtle':           ['Myrtle'],
    'Nearly Headless Nick':     ['Nick', 'Nicholas', 'Nearly Headless'],
    'Sir Patrick Delaney-Podmore': ['Patrick', 'Delaney-Podmore'],
    'The Bloody Baron':         ['Baron'],
    'Professor Dippet':         ['Dippet', 'Armando'],
    'Godric Gryffindor':        ['Godric'],
    'Salazar Slytherin':        ['Salazar'],
    'Professor Binns':          ['Binns'],
    'Professor Flitwick':       ['Flitwick', 'Filius'],
    'Professor McGonagall':     ['McGonagall', 'Minerva'],
    'Professor Sprout':         ['Sprout', 'Pomona'],
    'Madam Hooch':              ['Hooch'],
    'Madam Pince':              ['Pince'],
    'Madam Pomfrey':            ['Pomfrey', 'Poppy'],
    'Colin Creevey':            ['Colin', 'Creevey'],
    'Ernie Macmillan':          ['Ernie', 'Macmillan'],
    'Justin Finch-Fletchley':   ['Justin', 'Finch-Fletchley'],
    'Hannah Abbott':            ['Hannah', 'Abbott'],
    'Seamus Finnigan':          ['Seamus', 'Finnigan'],
    'Dean Thomas':              ['Dean'],
    'Millicent Bulstrode':      ['Millicent', 'Bulstrode'],
    'Vincent Crabbe':           ['Crabbe', 'Vincent'],
    'Gregory Goyle':            ['Goyle', 'Gregory'],
    'Katie Bell':               ['Katie', 'Bell'],
    'Angelina Johnson':         ['Angelina', 'Johnson'],
    'Alicia Spinnet':           ['Alicia', 'Spinnet'],
    'Oliver Wood':              ['Wood', 'Oliver'],
    'Fred Weasley':             ['Fred'],
    'George Weasley':           ['George'],
    'Bill Weasley':             ['Bill'],
    'Charlie Weasley':          ['Charlie'],
    'Percy Weasley':            ['Percy'],
    'Arthur Weasley':           ['Arthur', 'Mr Weasley', 'Mr. Weasley'],
    'Molly Weasley':            ['Molly', 'Mrs Weasley', 'Mrs. Weasley'],
    'The Masons':               ['Mason'],
    'Hedwig':                   ['Hedwig'],
    'Errol':                    ['Errol'],
    'Fang':                     ['Fang'],
    'Fawkes':                   ['Fawkes'],
    'Aragog':                   ['Aragog'],
    'Mosag':                    ['Mosag'],
    'Trevor':                   ['Trevor'],
    'Scabbers':                 ['Scabbers'],
    'Peeves':                   ['Peeves'],
    'The Headless Hunt':        ['Headless Hunt', 'Headless'],
}

print(f'{len(CHARACTERS)} personnages définis')

65 personnages définis


### 3.2 Définition des fonctions de comptage et cooccurrences

In [14]:
# La fonction ci-dessous compte les mentions en évitant le double comptage.
# Les alias multi-mots (ex. 'Harry Potter') sont prioritaires sur les alias courts ('Harry', 'Potter').
def compter_mentions(texte, personnages):
    counts = {}
    for perso, alias in personnages.items():
        alias_tries = sorted(alias, key=len, reverse=True)
        texte_restant = texte
        total = 0
        for a in alias_tries:
            matches = list(re.finditer(r'\b' + re.escape(a) + r'\b', texte_restant, re.IGNORECASE))
            total += len(matches)
            for m in reversed(matches):
                texte_restant = texte_restant[:m.start()] + ' ' * (m.end() - m.start()) + texte_restant[m.end():]
        if total > 0:
            counts[perso] = total
    return dict(sorted(counts.items(), key=lambda x: -x[1]))



# La fonction ci-dessous calcule les cooccurrences entre personnages par fenêtre glissante.
# On garde la même logique anti-double-comptage : alias triés du plus long au plus court, positions déjà consommées ignorées.
def get_cooccurrences(texte, personnages, fenetre=100):
    mots = texte.split()
    mentions = []  # liste de (index_mot, nom_canonique)

    for perso, alias in personnages.items():
        alias_tries = sorted(alias, key=len, reverse=True)
        positions_utilisees = set()

        for a in alias_tries:
            a_mots = a.split()
            n = len(a_mots)
            for i in range(len(mots) - n + 1):
                # Vérifier que ces positions ne sont pas déjà consommées
                if any(i + k in positions_utilisees for k in range(n)):
                    continue
                # Vérifier la correspondance mot à mot (insensible à la casse)
                if all(re.fullmatch(re.escape(a_mots[k]), mots[i + k], re.IGNORECASE) for k in range(n)):
                    mentions.append((i, perso))
                    for k in range(n):
                        positions_utilisees.add(i + k)

    # Trier par position dans le texte
    mentions.sort(key=lambda x: x[0])

    # Calcul des cooccurrences par fenêtre glissante
    aretes = collections.Counter()
    for idx_a, (pos_a, perso_a) in enumerate(mentions):
        for pos_b, perso_b in mentions[idx_a + 1:]:
            if pos_b - pos_a > fenetre:
                break
            if perso_a != perso_b:
                paire = tuple(sorted([perso_a, perso_b]))
                aretes[paire] += 1
    return aretes

### 3.3 Calcul des fréquences et cooccurrences

In [15]:
with open('outputs/novel_cleaned.txt', encoding='utf-8') as f:
    roman_propre = f.read()

with open('outputs/screenplay_cleaned.txt', encoding='utf-8') as f:
    script_propre = f.read()

print('Calcul des fréquences...')
freq_roman  = compter_mentions(roman_propre,  CHARACTERS)
freq_script = compter_mentions(script_propre, CHARACTERS)

print(f'\n{"Personnage":<30} {"Roman":>7}  {"Film":>6}')
print('-' * 48)
for perso in CHARACTERS:
    n = freq_roman.get(perso, 0)
    s = freq_script.get(perso, 0)
    if n > 0 or s > 0:
        print(f'{perso:<30} {n:>7}  {s:>6}')

print('\nCalcul des cooccurrences (peut prendre ~1 minute)...')
cooc_roman  = get_cooccurrences(roman_propre,  CHARACTERS, fenetre=100)
cooc_script = get_cooccurrences(script_propre, CHARACTERS, fenetre=100)

print(f'\nRoman  : {len(cooc_roman)} paires cooccurrentes')
print(f'Script : {len(cooc_script)} paires cooccurrentes')

Calcul des fréquences...

Personnage                       Roman    Film
------------------------------------------------
Harry Potter                      2010     876
Hermione Granger                   343     194
Ronald Weasley                     707     426
Ginny Weasley                      118      58
Albus Dumbledore                   166     104
Lord Voldemort                     168      99
Neville Longbottom                  40      20
Draco Malfoy                       252     142
Severus Snape                      103      34
Rubeus Hagrid                      167     112
Vernon Dursley                      41      38
Petunia Dursley                     29      11
Dudley Dursley                      36      15
Dobby                              165     106
Cornelius Fudge                     34      13
Lucius Malfoy                       31      56
Mr Borgin                           18      12
Mrs Norris                          24      10
Argus Filch                     

### 3.4 Affectation des maisons 

In [16]:
MAISONS = {
    'Harry Potter':                 'Gryffindor',
    'Hermione Granger':             'Gryffindor',
    'Ronald Weasley':               'Gryffindor',
    'Ginny Weasley':                'Gryffindor',
    'Neville Longbottom':           'Gryffindor',
    'Seamus Finnigan':              'Gryffindor',
    'Dean Thomas':                  'Gryffindor',
    'Colin Creevey':                'Gryffindor',
    'Katie Bell':                   'Gryffindor',
    'Angelina Johnson':             'Gryffindor',
    'Alicia Spinnet':               'Gryffindor',
    'Oliver Wood':                  'Gryffindor',
    'Fred Weasley':                 'Gryffindor',
    'George Weasley':               'Gryffindor',
    'Percy Weasley':                'Gryffindor',
    'Bill Weasley':                 'Gryffindor',
    'Charlie Weasley':              'Gryffindor',
    'Arthur Weasley':               'Gryffindor',
    'Molly Weasley':                'Gryffindor',
    'Albus Dumbledore':             'Gryffindor',
    'Professor McGonagall':         'Gryffindor',  
    'Rubeus Hagrid':                'Gryffindor',
    'Nearly Headless Nick':         'Gryffindor',
    'Godric Gryffindor':            'Gryffindor',
    'Draco Malfoy':                 'Slytherin',
    'Vincent Crabbe':               'Slytherin',
    'Gregory Goyle':                'Slytherin',
    'Millicent Bulstrode':          'Slytherin',
    'Severus Snape':                'Slytherin',
    'The Bloody Baron':             'Slytherin',
    'Lord Voldemort':               'Slytherin',
    'Lucius Malfoy':                'Slytherin',
    'Narcissa Malfoy':              'Slytherin',
    'Salazar Slytherin':            'Slytherin',
    'Justin Finch-Fletchley':       'Hufflepuff',
    'Ernie Macmillan':              'Hufflepuff',
    'Hannah Abbott':                'Hufflepuff',
    'Professor Sprout':             'Hufflepuff',  
    'Professor Flitwick':           'Ravenclaw',   
    'Moaning Myrtle':               'Ravenclaw',
    'Professor Binns':              'Other',
    'Professor Dippet':             'Other',
    'Argus Filch':                  'Other',
    'Madam Hooch':                  'Other',
    'Madam Pince':                  'Other',
    'Madam Pomfrey':                'Other',
    'Cornelius Fudge':              'Other',
    'Vernon Dursley':               'Other',
    'Petunia Dursley':              'Other',
    'Dudley Dursley':               'Other',
    'Mr Borgin':                    'Other',
    'The Masons':                   'Other',
    'Sir Patrick Delaney-Podmore':  'Other',
    'Dobby':                        'Other',
    'Peeves':                       'Other',
    'The Headless Hunt':            'Other',
    'Mrs Norris':                   'Other',
    'Hedwig':                       'Other',
    'Errol':                        'Other',
    'Fang':                         'Other',
    'Fawkes':                       'Other',
    'Aragog':                       'Other',
    'Mosag':                        'Other',
    'Trevor':                       'Other',
    'Scabbers':                     'Other',
}

### 3.5 Export pour Gephi 

In [17]:
### 4.1 Création du corpus pour Orange

# Orange attend un CSV avec au minimum :
# - une colonne 'text'  : le contenu textuel
# - une colonne 'label' : l'étiquette du document (roman / film)
# On crée deux versions :
# - une par document entier (pour comparer roman vs film globalement)
# - une par chapitre du roman (pour analyser l'évolution thématique)

# --- Version 1 : document entier ---
with open('outputs/novel_cleaned.txt', encoding='utf-8') as f:
    roman_propre = f.read()

with open('outputs/screenplay_cleaned.txt', encoding='utf-8') as f:
    script_propre = f.read()

df_corpus = pd.DataFrame([
    {'label': 'roman', 'text': roman_propre},
    {'label': 'film',  'text': script_propre},
])
df_corpus.to_csv('outputs/orange_corpus.csv', index=False, encoding='utf-8')
print('outputs/orange_corpus.csv  (2 documents : roman + film)')

# --- Version 2 : un document par chapitre du roman ---
df_chapitres = pd.DataFrame([
    {'label': f'roman_chap_{i:02d}', 'title': title, 'text': text}
    for i, (title, text) in enumerate(chapters.items(), 1)
])
df_chapitres.to_csv('outputs/orange_chapitres.csv', index=False, encoding='utf-8')
print('outputs/orange_chapitres.csv  (18 documents : un par chapitre)')

outputs/orange_corpus.csv  (2 documents : roman + film)
outputs/orange_chapitres.csv  (18 documents : un par chapitre)


In [18]:
# Sous la forme :
# Colonnes nodes : Id | Label | Weight | House
# Colonnes edges : Source | Target | Weight | Type

def exporter_gephi(freq, cooc, prefixe):
    chemin_nodes = f'outputs/{prefixe}_nodes.csv'
    chemin_edges = f'outputs/{prefixe}_edges.csv'

    with open(chemin_nodes, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Id', 'Label', 'Weight', 'House'])
        for perso, n in freq.items():
            maison = MAISONS.get(perso, 'Other')
            writer.writerow([perso, perso, n, maison])

    with open(chemin_edges, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Source', 'Target', 'Weight', 'Type'])
        for (a, b), poids in cooc.items():
            if a in freq and b in freq:
                writer.writerow([a, b, poids, 'Undirected'])

    print(f'[{prefixe}]  {len(freq)} nœuds  |  {len(cooc)} arêtes')
    print(f'  -> {chemin_nodes}')
    print(f'  -> {chemin_edges}')


exporter_gephi(freq_roman,  cooc_roman,  'novel')
print()
exporter_gephi(freq_script, cooc_script, 'screenplay')

[novel]  63 nœuds  |  746 arêtes
  -> outputs/novel_nodes.csv
  -> outputs/novel_edges.csv

[screenplay]  54 nœuds  |  509 arêtes
  -> outputs/screenplay_nodes.csv
  -> outputs/screenplay_edges.csv


---
## 4. Préparation pour Orange — Text Mining

Les fichiers texte produits dans les sections 1 et 2 servent ici de base pour l'analyse thématique dans Orange Data Mining. L'objectif est de comparer le **contenu lexical** du roman et du film : quels mots sont caractéristiques de l'un et pas de l'autre, et quels thèmes émergent de chaque corpus.

On exporte deux fichiers CSV lisibles directement par Orange :
- **`orange_corpus.csv`** : 2 documents (roman entier vs film entier) — pour la comparaison globale
- **`orange_chapitres.csv`** : 18 documents (un par chapitre) — pour observer l'évolution thématique au fil du roman

In [19]:
# Orange attend un CSV avec au minimum :
# - une colonne 'text'  : le contenu textuel
# - une colonne 'label' : l'étiquette du document (roman / film)
# On crée deux versions :
# - une par document entier (pour comparer roman vs film globalement)
# - une par chapitre du roman (pour analyser l'évolution thématique)

with open('outputs/novel_cleaned.txt', encoding='utf-8') as f:
    roman_propre = f.read()

with open('outputs/screenplay_cleaned.txt', encoding='utf-8') as f:
    script_propre = f.read()

df_corpus = pd.DataFrame([
    {'label': 'roman', 'text': roman_propre},
    {'label': 'film',  'text': script_propre},
])
df_corpus.to_csv('outputs/orange_corpus.csv', index=False, encoding='utf-8')
print('outputs/orange_corpus.csv  (2 documents : roman + film)')

outputs/orange_corpus.csv  (2 documents : roman + film)
